<a href="https://colab.research.google.com/github/eyasu-taye/About-Me/blob/main/td_similarity_nov_29_started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Single Colab cell: Gradio UI + Offline Embeddings + BERT embeddings + Phonetic + Fuzzy matching
# Paste into Colab and run.

# Install required packages (first-time only)
!pip install -q sentence-transformers gradio annoy rapidfuzz jellyfish unidecode

import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from annoy import AnnoyIndex
from rapidfuzz import fuzz
import jellyfish
from unidecode import unidecode
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------
# 0. Config
# -------------------------
EMB_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"  # multilingual, compact
EMB_DIM = 384  # embedding dim for the model above (checked by loading)
ANN_INDEX_PATH = "/content/drive/MyDrive/td_similarity/trade_ann.ann"
EMB_SAVE_PATH = "/content/drive/MyDrive/td_similarity/trade_embeddings.npy"
NAMES_SAVE_PATH = "/content/drive/MyDrive/td_similarity/trade_names.npy"
ANNOY_TREES = 10

# -------------------------
# 1. Sample dataset (replace with your CSV/Excel if you want)
# -------------------------
# If you have a CSV: uncomment and set filename
# df = pd.read_csv("/content/drive/MyDrive/td_similarity/your_trade_names.csv")

data = {
    "trade_name": [
        "ማኑፋክቸሪንግ ካምፓኒ",
        "አፍሪእሸቱ አ.ማ",
        "አብሮአዲስ የህብረት ስራ ማህበር",
        "ቴክሱራፌል ሶልዩሽን",
        "አፍሪእናተ ሶልዩሽን",
        "አፍሪያህንፃ አክሲዮን ማህበር",
        "ኢትዮተቋራጭ አ.ማ",
        "ሰነእዚህ ቢዝነስ ሴንተር",
        "በዚህ ሰርቪስ"
    ],
    "registration_status": [
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
        "የተመዘገበ",
    ]
}
df = pd.DataFrame(data)

# -------------------------
# 2. Load SentenceTransformer
# -------------------------
print("Loading embedding model...", EMB_MODEL_NAME)
embedder = SentenceTransformer(EMB_MODEL_NAME)
# get real embedding dim
try:
    EMB_DIM = embedder.get_sentence_embedding_dimension()
except:
    EMB_DIM = EMB_DIM
print("Embedding dim:", EMB_DIM)

# -------------------------
# 3. Build or load offline embeddings + Annoy index
# -------------------------
def build_and_save_embeddings(df, force_rebuild=False):
    # If files exist and not forcing, load them
    if (not force_rebuild) and os.path.exists(EMB_SAVE_PATH) and os.path.exists(ANN_INDEX_PATH) and os.path.exists(NAMES_SAVE_PATH):
        print("Embeddings and index already exist. Loading from disk.")
        names = np.load(NAMES_SAVE_PATH, allow_pickle=True)
        embeddings = np.load(EMB_SAVE_PATH)
        # load Annoy index
        t = AnnoyIndex(EMB_DIM, 'angular')
        t.load(ANN_INDEX_PATH)
        return names.tolist(), embeddings, t

    print("Computing embeddings and building Annoy index (this may take a moment)...")
    names = df["trade_name"].astype(str).tolist()
    embeddings = embedder.encode(names, show_progress_bar=True, convert_to_numpy=True)

    # save embeddings & names
    np.save(EMB_SAVE_PATH, embeddings)
    np.save(NAMES_SAVE_PATH, np.array(names, dtype=object))

    # build Annoy index
    t = AnnoyIndex(EMB_DIM, 'angular')
    for i, emb in enumerate(embeddings):
        t.add_item(i, emb.tolist())
    t.build(ANNOY_TREES)
    t.save(ANN_INDEX_PATH)
    print("Saved embeddings and Annoy index.")
    return names, embeddings, t

names_list, embeddings_np, annoy_index = build_and_save_embeddings(df)

# -------------------------
# 4. Helper functions: similarity, phonetic, fuzzy
# -------------------------
def cos_sim(a, b):
    # robust cosine
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def phonetic_code(text):
    # transliterate to ASCII then apply Double Metaphone
    tr = unidecode(str(text))
    dm = jellyfish.metaphone(tr) if hasattr(jellyfish, "metaphone") else jellyfish.double_metaphone(tr)[0]
    # jellyfish.metaphone may be present; double_metaphone returns tuple
    if dm is None:
        dm = ""
    return dm

def phonetic_score(a, b):
    # simple: equal phonetic codes -> 100, else 0
    ca = phonetic_code(a)
    cb = phonetic_code(b)
    if ca and cb and ca == cb:
        return 100.0
    # fallback: compute token-based fuzzy on transliterated strings
    return fuzz.token_set_ratio(unidecode(str(a)), unidecode(str(b)))

# -------------------------
# 5. Main lookup function used by UI
# -------------------------
def lookup_trade_name(query,
                      top_k=5,
                      sim_threshold=85.0,
                      require_registered_for_reject=True):
    query = str(query)
    # get embedding
    q_emb = embedder.encode([query], convert_to_numpy=True)[0]

    # query Annoy for top_k candidates
    idxs = annoy_index.get_nns_by_vector(q_emb.tolist(), top_k, include_distances=False)

    results = []
    for idx in idxs:
        candidate = names_list[idx]
        emb = embeddings_np[idx]
        sim = cos_sim(q_emb, emb) * 100.0  # percentage
        fuzzy_ratio = fuzz.token_set_ratio(query, candidate)  # 0-100
        phon_score = phonetic_score(query, candidate)  # 0-100

        reg_status = df.loc[df["trade_name"] == candidate, "registration_status"]
        reg_status = reg_status.values[0] if len(reg_status) > 0 else ""

        results.append({
            "trade_name": candidate,
            "registration_status": reg_status,
            "embedding_similarity_pct": round(sim, 2),
            "fuzzy_pct": round(fuzzy_ratio, 2),
            "phonetic_pct": round(phon_score, 2)
        })

    # Sort by embedding similarity
    results = sorted(results, key=lambda x: x["embedding_similarity_pct"], reverse=True)

    # Decide rejection:
    # If any candidate has embedding_similarity >= sim_threshold AND (if require_registered_for_reject: candidate registered)
    reject_reasons = []
    for r in results:
        if r["embedding_similarity_pct"] >= sim_threshold:
            if require_registered_for_reject:
                if str(r["registration_status"]).strip() == "የተመዘገበ":
                    reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
            else:
                reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))

    decision = "ACCEPTED"
    message = f"No matches \u2265 {sim_threshold:.1f}% (registered) found."
    if len(reject_reasons) > 0:
        decision = "REJECTED"
        message = f"Found {len(reject_reasons)} registered name(s) \u2265 {sim_threshold:.1f}% similarity."

    # Build a pandas DataFrame for display
    out_df = pd.DataFrame(results)
    return decision, message, out_df

# -------------------------
# 6. Gradio UI
# -------------------------
def gradio_lookup(query, top_k=5, sim_threshold=85.0, require_registered=True):
    decision, message, out_df = lookup_trade_name(query, top_k=int(top_k), sim_threshold=float(sim_threshold), require_registered_for_reject=require_registered)
    # format table for display
    table_html = out_df.to_html(index=False)
    return decision, message, table_html

with gr.Blocks() as demo:
    gr.Markdown("## Trade Name Similarity Checker \u2014 BERT embeddings + Phonetic + Fuzzy + Offline Search")
    with gr.Row():
        with gr.Column(scale=2):
            query = gr.Textbox(lines=2, label="Input trade name", placeholder="Enter trade name to check...")
            top_k = gr.Slider(minimum=1, maximum=20, value=5, step=1, label="Top K candidates")
            sim_threshold = gr.Slider(minimum=50, maximum=100, value=85, step=1, label="Reject if embedding similarity \u2265 (percent)")
            require_registered = gr.Checkbox(value=True, label="Only reject if matching name is registered (\u12e8\u12f0\u121b\u122b\u12f0\u121b\u12f0)")
            run_btn = gr.Button("Check")
            rebuild_btn = gr.Button("Rebuild embeddings & index")
        with gr.Column(scale=3):
            decision_out = gr.Textbox(label="Decision", interactive=False)
            message_out = gr.Textbox(label="Message", interactive=False)
            results_html = gr.HTML()
    # actions
    run_btn.click(fn=gradio_lookup, inputs=[query, top_k, sim_threshold, require_registered], outputs=[decision_out, message_out, results_html])
    def rebuild_action(): # Changed from rebuild_action(_):
        # rebuild embeddings and index from current df
        global names_list, embeddings_np, annoy_index # Ensure global variables are updated
        names_list, embeddings_np, annoy_index = build_and_save_embeddings(df, force_rebuild=True)
        return "Rebuilt embeddings & index."
    rebuild_btn.click(fn=rebuild_action, inputs=[], outputs=[message_out])

# Launch the app (in Colab you'll get a public link)
demo.launch(share=True)

Loading embedding model... paraphrase-multilingual-MiniLM-L12-v2
Embedding dim: 384
Embeddings and index already exist. Loading from disk.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d334ffbba45b61e030.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
# Multi-Model Trainer + Per-epoch Progress Streaming + SBERT + Phonetic + Gradio UI
# Single-file ready for Colab or local notebook. This enhances your original script to:
# - Train *all* models sequentially
# - Stream per-epoch progress back to Gradio via a generator (so the UI updates live)
# - Save per-model training plots and a summary CSV containing final metrics for comparison
# - Provide two Gradio actions: Train single model (as before) and Train ALL models (streaming)

# NOTE: run this entire file in one Colab cell (or a Python environment with display support).
# Make sure you mount Google Drive if you want saved models under /content/drive.

# --------- Install required packages (uncomment if running first time) ---------
# !pip install -q sentence-transformers gradio gensim annoy jellyfish unidecode rapidfuzz matplotlib

# ------------------------------ Imports -------------------------------------
import os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, LSTM, Bidirectional, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sentence_transformers import SentenceTransformer
from gensim.models import FastText
from annoy import AnnoyIndex
from rapidfuzz import fuzz
import jellyfish
from unidecode import unidecode
import gradio as gr
import pickle

# ------------------------------ Config --------------------------------------
MODEL_NAMES = [
    "cnn_word","cnn_char","cnn_combined",
    "cnn_fasttext_keras","cnn_fasttext_gensim",
    "rnn_word","rnn_char","rnn_combined",
    "rnn_fasttext_keras","rnn_fasttext_gensim"
]
DEFAULT_EPOCHS = 5
FT_DIM = 50
KERAS_EMB_DIM = 64
SAVED_MODELS_DIR = "/content/drive/MyDrive/TradeNameSimilarity/Models"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
PLOTS_DIR = "/content/drive/MyDrive/TradeNameSimilarity/Models/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)
METRICS_CSV = os.path.join(PLOTS_DIR, "all_models_summary.csv")

# ----------------------------- Sample data ----------------------------------
sample_data = {
    "trade_name": [
        "ማኑፋክቸሪንግ ካምፓኒ",
        "አፍሪእሸቱ አ.ማ",
        "አብሮአዲስ የህብረት ስራ ማህበር",
        "ቴክሱራፌል ሶልዩሽን",
        "አፍሪእናተ ሶልዩሽን",
        "አፍሪያህንፃ አክሲዮን ማህበር",
        "ኢትዮተቋራጭ አ.ማ",
        "ሰነእዚህ ቢዝነስ ሴንተር",
        "በዚህ ሰርቪስ",
        "አዲስ ንግድ ስም"
    ],
    "registration_status": [
        "የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ",
        "የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ","አልተመዘገበ"
    ]
}

df = pd.DataFrame(sample_data)
df['label'] = df['registration_status'].apply(lambda x: 1 if str(x).strip()=="የተመዘገበ" else 0)

# --------------------------- Preprocessing ----------------------------------
texts = df['trade_name'].astype(str).tolist()

# WORD tokenizer
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(texts)
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1
MAX_WORD_LEN = 20

def encode_word(s):
    seq = tokenizer.texts_to_sequences([str(s)])
    return pad_sequences(seq, maxlen=MAX_WORD_LEN)[0]

# CHAR mapping
all_chars = sorted(list(set(" ".join(texts))))
char2idx = {c:i+1 for i,c in enumerate(all_chars)}
MAX_CHAR_LEN = 40

def encode_char(s):
    s = str(s)[:MAX_CHAR_LEN]
    arr = [char2idx.get(c,0) for c in s]
    if len(arr) < MAX_CHAR_LEN:
        arr += [0]*(MAX_CHAR_LEN - len(arr))
    return np.array(arr)

# FastText (gensim) training
ft_sentences = [t.split() for t in texts]
fasttext_model = FastText(sentences=ft_sentences, vector_size=FT_DIM, window=3, min_count=1, epochs=20)

# Pre-init embedding matrix for Keras-fasttext option
embedding_matrix = np.zeros((VOCAB_SIZE, FT_DIM))
for word, i in word_index.items():
    if i < VOCAB_SIZE:
        if word in fasttext_model.wv:
            embedding_matrix[i] = fasttext_model.wv[word]
        else:
            embedding_matrix[i] = np.random.uniform(-0.25, 0.25, FT_DIM)

# Gensim FT seq builder
def texts_to_ft_seq_array(texts_list):
    seqs = []
    for t in texts_list:
        toks = str(t).split()[:MAX_WORD_LEN]
        vecs = []
        for token in toks:
            if token in fasttext_model.wv:
                vecs.append(fasttext_model.wv[token])
            else:
                vecs.append(np.zeros(FT_DIM))
        while len(vecs) < MAX_WORD_LEN:
            vecs.append(np.zeros(FT_DIM))
        seqs.append(np.array(vecs))
    return np.array(seqs)

# Precompute inputs
X_word = np.array([encode_word(t) for t in texts])
X_char = np.array([encode_char(t) for t in texts])
X_ft_seqs = texts_to_ft_seq_array(texts)
y = df['label'].values

Xw_train, Xw_test, Xc_train, Xc_test, Xft_train, Xft_test, y_train, y_test = train_test_split(
    X_word, X_char, X_ft_seqs, y, test_size=0.2, random_state=42
)

# ---------------------------- Model factory ---------------------------------
def build_model(model_name, lr=1e-3):
    if model_name == "cnn_word":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, 128)(inp)
        x = Conv1D(128, 3, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_char":
        inp = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        x = Embedding(len(char2idx)+1, 64)(inp)
        x = Conv1D(128, 5, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_combined":
        in_w = Input(shape=(MAX_WORD_LEN,), name="word_input")
        in_c = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        w = Embedding(VOCAB_SIZE, 128)(in_w)
        w = Conv1D(64, 3, activation="relu")(w)
        w = GlobalMaxPooling1D()(w)
        c = Embedding(len(char2idx)+1, 64)(in_c)
        c = Conv1D(64, 5, activation="relu")(c)
        c = GlobalMaxPooling1D()(c)
        x = Concatenate()([w,c])
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model([in_w, in_c], out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_fasttext_keras":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, FT_DIM, weights=[embedding_matrix], trainable=True, name="keras_ft_emb")(inp)
        x = Conv1D(128, 3, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_fasttext_gensim":
        inp = Input(shape=(MAX_WORD_LEN, FT_DIM), name="ft_seq_input")
        x = Conv1D(128, 3, activation="relu")(inp)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_word":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, 128)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_char":
        inp = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        x = Embedding(len(char2idx)+1, 64)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_combined":
        in_w = Input(shape=(MAX_WORD_LEN,), name="word_input")
        in_c = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        w = Embedding(VOCAB_SIZE, 128)(in_w)
        w = Bidirectional(LSTM(64))(w)
        c = Embedding(len(char2idx)+1, 64)(in_c)
        c = Bidirectional(LSTM(64))(c)
        x = Concatenate()([w,c])
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model([in_w, in_c], out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_fasttext_keras":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, FT_DIM, weights=[embedding_matrix], trainable=True)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_fasttext_gensim":
        inp = Input(shape=(MAX_WORD_LEN, FT_DIM), name="ft_seq_input")
        x = Bidirectional(LSTM(64))(inp)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    raise ValueError(f"Unsupported model: {model_name}")

# ------------------------- Train model (per-epoch) ---------------------------
# This function trains per-epoch so we can yield progress back to a caller.
def evaluate_on_test(model_obj, model_name):
    """Evaluate trained model on the holdout test set and return metrics dict."""
    # select test inputs depending on model type
    if model_name in ["cnn_word","rnn_word","cnn_fasttext_keras","rnn_fasttext_keras"]:
        X_test_local = Xw_test
    elif model_name in ["cnn_char","rnn_char"]:
        X_test_local = Xc_test
    elif model_name in ["cnn_combined","rnn_combined"]:
        X_test_local = [Xw_test, Xc_test]
    elif model_name in ["cnn_fasttext_gensim","rnn_fasttext_gensim"]:
        X_test_local = Xft_test
    else:
        X_test_local = Xw_test

    # predict probabilities
    preds_proba = model_obj.predict(X_test_local, verbose=0)
    # ensure shape
    preds_proba = np.array(preds_proba).reshape(-1)
    preds = (preds_proba >= 0.5).astype(int)

    acc = float(accuracy_score(y_test, preds)) if len(y_test) > 0 else 0.0
    prec = float(precision_score(y_test, preds, zero_division=0))
    rec = float(recall_score(y_test, preds, zero_division=0))
    f1 = float(f1_score(y_test, preds, zero_division=0))
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


def train_model_per_epoch(model_name, epochs=DEFAULT_EPOCHS, batch_size=16, lr=1e-3, verbose=False):
    model = build_model(model_name, lr=lr)
    full_history = {"loss":[], "val_loss":[], "accuracy":[], "val_accuracy":[]}

    # Select inputs
    if model_name in ["cnn_word","rnn_word","cnn_fasttext_keras","rnn_fasttext_keras"]:
        X_train = Xw_train
    elif model_name in ["cnn_char","rnn_char"]:
        X_train = Xc_train
    elif model_name in ["cnn_combined","rnn_combined"]:
        X_train = [Xw_train, Xc_train]
    elif model_name in ["cnn_fasttext_gensim","rnn_fasttext_gensim"]:
        X_train = Xft_train
    else:
        X_train = Xw_train

    for e in range(1, int(epochs)+1):
        # train for 1 epoch with validation_split=0.1
        hist = model.fit(X_train, y_train, validation_split=0.1, epochs=1, batch_size=int(batch_size), verbose=0)
        # aggregate
        for k,v in hist.history.items():
            if k not in full_history: full_history[k] = []
            full_history[k].append(v[0])
        # yield per-epoch metrics
        yield {
            "model": model_name,
            "epoch": e,
            "loss": float(full_history.get('loss')[-1]) if full_history.get('loss') else None,
            "val_loss": float(full_history.get('val_loss')[-1]) if full_history.get('val_loss') else None,
            "accuracy": float(full_history.get('accuracy')[-1]) if full_history.get('accuracy') else None,
            "val_accuracy": float(full_history.get('val_accuracy')[-1]) if full_history.get('val_accuracy') else None,
            "finished": False
        }

    # Save model at the end
    model_path = os.path.join(SAVED_MODELS_DIR, f"{model_name}.h5")
    model.save(model_path)

    # Evaluate on holdout test set
    metrics = evaluate_on_test(model, model_name)

    # final yield marking finished (no plot generation)
    yield {
        "model": model_name,
        "epoch": int(epochs),
        "loss": float(full_history.get('loss')[-1]) if full_history.get('loss') else None,
        "val_loss": float(full_history.get('val_loss')[-1]) if full_history.get('val_loss') else None,
        "accuracy": float(full_history.get('accuracy')[-1]) if full_history.get('accuracy') else None,
        "val_accuracy": float(full_history.get('val_accuracy')[-1]) if full_history.get('val_accuracy') else None,
        "finished": True,
        "model_path": model_path,
        "metrics": metrics,
        "history": full_history
    }

# ---------------------- Train ALL models generator --------------------------
# This generator trains each model sequentially and yields UI-updates for Gradio.
def train_all_models_generator(epochs=DEFAULT_EPOCHS, batch_size=16, lr=1e-3):
    # prepare summary list
    summary_rows = []
    # iterate
    for model_name in MODEL_NAMES:
        # Yield message that this model will start
        yield (f"Starting training: {model_name}", "", "", "", "", "")
        # loop per-epoch yields
        for epoch_info in train_model_per_epoch(model_name, epochs=epochs, batch_size=batch_size, lr=lr):
            # Compose train_info text and saved model listing (saved after finish)
            if not epoch_info.get('finished'):
                train_info = f"{epoch_info['model']} - epoch {epoch_info['epoch']}/{epochs} | acc={epoch_info.get('accuracy'):.4f} val_acc={epoch_info.get('val_accuracy'):.4f}"
                saved_list = "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')])
                metrics_html = ""
                decision = ""
                message = ""
                table_html = ""
                yield (train_info, saved_list, metrics_html, decision, message, table_html)
            else:
                # finished - include metrics and update metrics summary
                model_path = epoch_info.get('model_path')
                saved_list = "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')])
                train_info = f"Finished {model_name} (epochs={epochs}). Saved: {os.path.basename(model_path)}"
                metrics = epoch_info.get('metrics', {})
                # append summary
                summary_rows.append({
                    'model': model_name,
                    'final_accuracy': metrics.get('accuracy'),
                    'final_precision': metrics.get('precision'),
                    'final_recall': metrics.get('recall'),
                    'final_f1': metrics.get('f1'),
                    'loss': epoch_info.get('loss'),
                    'val_loss': epoch_info.get('val_loss'),
                    'model_file': os.path.basename(model_path)
                })
                # save intermediate summary CSV
                pd.DataFrame(summary_rows).to_csv(METRICS_CSV, index=False)
                table_html = pd.DataFrame(summary_rows).to_html(index=False)
                metrics_html = table_html
                decision = ""
                message = ""
                yield (train_info, saved_list, metrics_html, decision, message, table_html)
    # All done - final message
    final_html = pd.read_csv(METRICS_CSV).to_html(index=False) if os.path.exists(METRICS_CSV) else ""
    yield ("ALL MODELS TRAINED", "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')]), "", "", "", final_html)

# ---------------- SBERT + phonetic + helpers (unchanged) -------------------
print("Loading SBERT (paraphrase-multilingual-MiniLM-L12-v2) — may take a moment ...")
try:
    sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
except Exception as e:
    print("Could not load SBERT model (offline?).", e)
    sbert = None

SBERT_DIM = sbert.get_sentence_embedding_dimension() if sbert is not None else 384

# phonetic and fuzzy

def phonetic_code(text):
    tr = unidecode(str(text))
    try:
        dm = jellyfish.double_metaphone(tr)[0]
    except:
        dm = jellyfish.metaphone(tr) if hasattr(jellyfish, "metaphone") else ""
    return dm or ""

def phonetic_score(a, b):
    ca = phonetic_code(a)
    cb = phonetic_code(b)
    if ca and cb and ca == cb:
        return 100.0
    return fuzz.token_set_ratio(unidecode(str(a)), unidecode(str(b)))

def cosine(a,b):
    a = np.array(a, dtype=float); b = np.array(b, dtype=float)
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na==0 or nb==0: return 0.0
    return float(np.dot(a,b)/(na*nb))

# embedding cache
emb_cache = {}

# helper to extract penultimate layer
def build_embedding_model_from_trained(model_obj, model_name):
    final_dense_idx = None
    for i, layer in enumerate(model_obj.layers[::-1]):
        if isinstance(layer, tf.keras.layers.Dense) and layer.output_shape[-1] == 1:
            final_dense_idx = len(model_obj.layers) - 1 - i
            break
    if final_dense_idx is None or final_dense_idx - 1 < 0:
        raise ValueError("Cannot find penultimate layer to extract embedding. Model structure might be unexpected.")
    embed_layer_output = model_obj.layers[final_dense_idx - 1].output
    return Model(inputs=model_obj.input, outputs=embed_layer_output)

# prepare embeddings for source
def prepare_embeddings_for_source(source_name):
    if source_name in emb_cache:
        return emb_cache[source_name]
    names = df['trade_name'].astype(str).tolist()
    embs = None
    if source_name == "bert_sentence":
        if sbert is None:
            raise ValueError("SBERT not loaded")
        embs = sbert.encode(names, convert_to_numpy=True, show_progress_bar=False)
    else:
        model_path = os.path.join(SAVED_MODELS_DIR, f"{source_name}.h5")
        if not os.path.exists(model_path):
            raise ValueError(f"Model '{source_name}' not found at {model_path}. Please train it first.")
        model_obj = load_model(model_path)
        emb_model = build_embedding_model_from_trained(model_obj, source_name)
        emb_list = []
        for name in names:
            if source_name in ["cnn_word", "rnn_word", "cnn_fasttext_keras", "rnn_fasttext_keras"]:
                x_input = np.array([encode_word(name)])
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_char", "rnn_char"]:
                x_input = np.array([encode_char(name)])
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_combined", "rnn_combined"]:
                x_input = [np.array([encode_word(name)]), np.array([encode_char(name)])]
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_fasttext_gensim", "rnn_fasttext_gensim"]:
                toks = str(name).split()[:MAX_WORD_LEN]
                seq = []
                for tkn in toks:
                    if tkn in fasttext_model.wv:
                        seq.append(fasttext_model.wv[tkn])
                    else:
                        seq.append(np.zeros(FT_DIM))
                while len(seq) < MAX_WORD_LEN:
                    seq.append(np.zeros(FT_DIM))
                x_input = np.array([seq])
                emb = emb_model.predict(x_input, verbose=0)[0]
            else:
                raise ValueError(f"Unknown source_name for embedding extraction: {source_name}")
            emb_list.append(emb)
        embs = np.vstack(emb_list)
    emb_cache[source_name] = (names, embs)
    return names, embs

# main lookup
def find_similar(query, source_name="bert_sentence", top_k=5, sim_threshold=85.0, require_registered_for_reject=True):
    query = str(query)
    q_emb = None
    if source_name == "bert_sentence":
        if sbert is None: raise ValueError("SBERT not loaded")
        q_emb = sbert.encode([query], convert_to_numpy=True)[0]
    else:
        model_path = os.path.join(SAVED_MODELS_DIR, f"{source_name}.h5")
        if not os.path.exists(model_path):
            raise ValueError(f"Model '{source_name}' not found at {model_path}. Please train it first.")
        model_obj = load_model(model_path)
        emb_model = build_embedding_model_from_trained(model_obj, source_name)
        if source_name in ["cnn_word", "rnn_word", "cnn_fasttext_keras", "rnn_fasttext_keras"]:
            x_input = np.array([encode_word(query)])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_char", "rnn_char"]:
            x_input = np.array([encode_char(query)])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_combined", "rnn_combined"]:
            x_input = [np.array([encode_word(query)]), np.array([encode_char(query)])]
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_fasttext_gensim", "rnn_fasttext_gensim"]:
            toks = str(query).split()[:MAX_WORD_LEN]
            seq = []
            for tkn in toks:
                if tkn in fasttext_model.wv:
                    seq.append(fasttext_model.wv[tkn])
                else:
                    seq.append(np.zeros(FT_DIM))
            while len(seq) < MAX_WORD_LEN:
                seq.append(np.zeros(FT_DIM))
            x_input = np.array([seq])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        else:
            raise ValueError(f"Unknown source_name for query embedding: {source_name}")

    names_list, embeddings_np = prepare_embeddings_for_source(source_name)
    results = []
    for idx, candidate in enumerate(names_list):
        emb = embeddings_np[idx]
        sim = cosine(q_emb, emb) * 100.0
        fuzzy_ratio = fuzz.token_set_ratio(query, candidate)
        phon_score = phonetic_score(query, candidate)
        reg_status = df.loc[df["trade_name"] == candidate, "registration_status"]
        reg_status = reg_status.values[0] if len(reg_status) > 0 else ""
        results.append({
            "trade_name": candidate,
            "registration_status": reg_status,
            "embedding_similarity_pct": round(sim, 2),
            "fuzzy_pct": round(fuzzy_ratio, 2),
            "phonetic_pct": round(phon_score, 2)
        })
    results = sorted(results, key=lambda x: x["embedding_similarity_pct"], reverse=True)
    top_k_results = results[:top_k]
    reject_reasons = []
    for r in top_k_results:
        if r["embedding_similarity_pct"] >= sim_threshold:
            if require_registered_for_reject:
                if str(r["registration_status"]).strip() == "የተመዘገበ":
                    reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
            else:
                reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
    decision = "ACCEPTED"
    message = f"No registered names ≥ {sim_threshold:.1f}% similarity found."
    if len(reject_reasons) > 0:
        decision = "REJECTED"
        message = f"Found {len(reject_reasons)} registered name(s) ≥ {sim_threshold:.1f}% similarity."
    out_df = pd.DataFrame(top_k_results)
    return decision, message, out_df

# -------------------- Utility: list saved models -----------------------------
def list_saved_models():
    files = [f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')]
    return files

# ------------------------- Gradio wiring ------------------------------------
# Single-model training wrapper (non-streaming) kept for compatibility

def ui_train_single_and_check(
    chosen_model, epochs, batch_size, lr, train_flag,
    check_query, source_for_check, top_k, sim_threshold, require_registered
):
    train_info = ""
    metrics_html = ""
    saved_list = "<br>".join(list_saved_models())
    if train_flag:
        try:
            # run full training (this will block until finish)
            final = None
            for epoch_info in train_model_per_epoch(chosen_model, epochs=epochs, batch_size=batch_size, lr=lr):
                final = epoch_info
            train_info = f"Trained {chosen_model}: epochs={epochs}, batch={batch_size}, lr={lr}"
            if final and final.get('metrics'):
                metrics = final.get('metrics')
                metrics_html = pd.DataFrame([{"metric":k, "value":v} for k,v in metrics.items()]).to_html(index=False)
            saved_list = "<br>".join(list_saved_models())
            if chosen_model in emb_cache: del emb_cache[chosen_model]
        except Exception as e:
            train_info = "TRAIN ERROR: " + str(e)

    decision = ""
    message = ""
    table_html = ""
    if str(check_query).strip():
        try:
            decision, message, out_df = find_similar(check_query, source_name=source_for_check, top_k=int(top_k), sim_threshold=float(sim_threshold), require_registered_for_reject=bool(require_registered))
            table_html = out_df.to_html(index=False)
        except Exception as e:
            decision = "ERROR"
            message = str(e)
            table_html = ""
    return train_info, saved_list, metrics_html, decision, message, table_html

# Train-all generator wrapper for Gradio streaming
def ui_train_all(epochs, batch_size, lr):
    # This is a generator that yields tuples matching our UI outputs
    for out in train_all_models_generator(epochs=int(epochs), batch_size=int(batch_size), lr=float(lr)):
        # out is tuple (train_info, saved_list, plot_html, decision, message, table_html)
        yield out

# Build Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## Trade Name Similarity System — Multi-Model Trainer\n\nUse 'Train All Models' to train every model sequentially and stream per-epoch progress.\nUse 'Train Single Model' to train only the selected model.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Training Controls")
            chosen_model = gr.Dropdown(MODEL_NAMES, value="cnn_word", label="Choose model to train")
            epochs = gr.Slider(minimum=1, maximum=50, step=1, value=5, label="Epochs")
            batch_size = gr.Slider(minimum=4, maximum=64, step=1, value=16, label="Batch size")
            lr = gr.Number(value=1e-3, label="Learning rate")
            train_flag = gr.Checkbox(label="Train model now?", value=False)
            train_btn = gr.Button("Run Training")
            train_all_btn = gr.Button("Train ALL Models (stream)")
            train_info_out = gr.Textbox(label="Train Info", interactive=False)
            saved_models_out = gr.HTML(label="Saved Models")
            train_plot_out = gr.HTML(label="Training Plot")

        with gr.Column(scale=1):
            gr.Markdown("### Similarity Check")
            check_query = gr.Textbox(lines=2, label="Input trade name to check")
            source_for_check_options = MODEL_NAMES + ["bert_sentence"]
            source_for_check = gr.Dropdown(source_for_check_options, value="bert_sentence", label="Source for Similarity Check")
            top_k = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="Top K candidates")
            sim_threshold = gr.Slider(minimum=50, maximum=100, step=1, value=85, label="Reject threshold (embedding %)")
            require_registered = gr.Checkbox(label="Only reject if candidate is registered (የተመዘገበ)", value=True)
            check_btn = gr.Button("Check Similarity")
            decision_out = gr.Textbox(label="Decision", interactive=False)
            message_out = gr.Textbox(label="Message", interactive=False)
            table_out = gr.HTML(label="Similarity Results")

    # Single model training (blocking)
    train_btn.click(
        fn=ui_train_single_and_check,
        inputs=[
            chosen_model, epochs, batch_size, lr,
            check_query, source_for_check, top_k, sim_threshold, require_registered
        ],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

    # Train all models (streaming generator) -> will update the same outputs incrementally
    train_all_btn.click(
        fn=ui_train_all,
        inputs=[epochs, batch_size, lr],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

    # Similarity check (non-training)
    check_btn.click(
        fn=ui_train_single_and_check,
        inputs=[
            chosen_model, epochs, batch_size, lr,
             # ensure train_flag False
            check_query, source_for_check, top_k, sim_threshold, require_registered
        ],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

# Launch (you can set share=True in Colab to obtain public link)
demo.launch(share=True)


Loading SBERT (paraphrase-multilingual-MiniLM-L12-v2) — may take a moment ...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73384c3203333aa025.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
# Multi-Model Trainer + Per-epoch Progress Streaming + SBERT + Phonetic + Gradio UI
# Single-file ready for Colab or local notebook. This enhances your original script to:
# - Train *all* models sequentially
# - Stream per-epoch progress back to Gradio via a generator (so the UI updates live)
# - Save per-model training plots and a summary CSV containing final metrics for comparison
# - Provide two Gradio actions: Train single model (as before) and Train ALL models (streaming)

# NOTE: run this entire file in one Colab cell (or a Python environment with display support).
# Make sure you mount Google Drive if you want saved models under /content/drive.

# --------- Install required packages (uncomment if running first time) ---------
# !pip install -q sentence-transformers gradio gensim annoy jellyfish unidecode rapidfuzz matplotlib

# ------------------------------ Imports -------------------------------------
import os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, LSTM, Bidirectional, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from gensim.models import FastText
from annoy import AnnoyIndex
from rapidfuzz import fuzz
import jellyfish
from unidecode import unidecode
import gradio as gr
import pickle

# ------------------------------ Config --------------------------------------
MODEL_NAMES = [
    "cnn_word","cnn_char","cnn_combined",
    "cnn_fasttext_keras","cnn_fasttext_gensim",
    "rnn_word","rnn_char","rnn_combined",
    "rnn_fasttext_keras","rnn_fasttext_gensim"
]
DEFAULT_EPOCHS = 5
FT_DIM = 50
KERAS_EMB_DIM = 64
SAVED_MODELS_DIR = "/content/drive/MyDrive/TradeNameSimilarity/Models_2"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
PLOTS_DIR = "/content/drive/MyDrive/TradeNameSimilarity/Models/plots_2"
os.makedirs(PLOTS_DIR, exist_ok=True)
METRICS_CSV = os.path.join(PLOTS_DIR, "all_models_summary.csv")

# ----------------------------- Sample data ----------------------------------
sample_data = {
    "trade_name": [
        "ማኑፋክቸሪንግ ካምፓኒ",
        "አፍሪእሸቱ አ.ማ",
        "አብሮአዲስ የህብረት ስራ ማህበር",
        "ቴክሱራፌል ሶልዩሽን",
        "አፍሪእናተ ሶልዩሽን",
        "አፍሪያህንፃ አክሲዮን ማህበር",
        "ኢትዮተቋራጭ አ.ማ",
        "ሰነእዚህ ቢዝነስ ሴንተር",
        "በዚህ ሰርቪስ",
        "አዲስ ንግድ ስም"
    ],
    "registration_status": [
        "የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ",
        "የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ","አልተመዘገበ"
    ]
}

df = pd.DataFrame(sample_data)
df['label'] = df['registration_status'].apply(lambda x: 1 if str(x).strip()=="የተመዘገበ" else 0)

# --------------------------- Preprocessing ----------------------------------
texts = df['trade_name'].astype(str).tolist()

# WORD tokenizer
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(texts)
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1
MAX_WORD_LEN = 20

def encode_word(s):
    seq = tokenizer.texts_to_sequences([str(s)])
    return pad_sequences(seq, maxlen=MAX_WORD_LEN)[0]

# CHAR mapping
all_chars = sorted(list(set(" ".join(texts))))
char2idx = {c:i+1 for i,c in enumerate(all_chars)}
MAX_CHAR_LEN = 40

def encode_char(s):
    s = str(s)[:MAX_CHAR_LEN]
    arr = [char2idx.get(c,0) for c in s]
    if len(arr) < MAX_CHAR_LEN:
        arr += [0]*(MAX_CHAR_LEN - len(arr))
    return np.array(arr)

# FastText (gensim) training
ft_sentences = [t.split() for t in texts]
fasttext_model = FastText(sentences=ft_sentences, vector_size=FT_DIM, window=3, min_count=1, epochs=20)

# Pre-init embedding matrix for Keras-fasttext option
embedding_matrix = np.zeros((VOCAB_SIZE, FT_DIM))
for word, i in word_index.items():
    if i < VOCAB_SIZE:
        if word in fasttext_model.wv:
            embedding_matrix[i] = fasttext_model.wv[word]
        else:
            embedding_matrix[i] = np.random.uniform(-0.25, 0.25, FT_DIM)

# Gensim FT seq builder
def texts_to_ft_seq_array(texts_list):
    seqs = []
    for t in texts_list:
        toks = str(t).split()[:MAX_WORD_LEN]
        vecs = []
        for token in toks:
            if token in fasttext_model.wv:
                vecs.append(fasttext_model.wv[token])
            else:
                vecs.append(np.zeros(FT_DIM))
        while len(vecs) < MAX_WORD_LEN:
            vecs.append(np.zeros(FT_DIM))
        seqs.append(np.array(vecs))
    return np.array(seqs)

# Precompute inputs
X_word = np.array([encode_word(t) for t in texts])
X_char = np.array([encode_char(t) for t in texts])
X_ft_seqs = texts_to_ft_seq_array(texts)
y = df['label'].values

Xw_train, Xw_test, Xc_train, Xc_test, Xft_train, Xft_test, y_train, y_test = train_test_split(
    X_word, X_char, X_ft_seqs, y, test_size=0.2, random_state=42
)

# ---------------------------- Model factory ---------------------------------
def build_model(model_name, lr=1e-3):
    if model_name == "cnn_word":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, 128)(inp)
        x = Conv1D(128, 3, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_char":
        inp = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        x = Embedding(len(char2idx)+1, 64)(inp)
        x = Conv1D(128, 5, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_combined":
        in_w = Input(shape=(MAX_WORD_LEN,), name="word_input")
        in_c = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        w = Embedding(VOCAB_SIZE, 128)(in_w)
        w = Conv1D(64, 3, activation="relu")(w)
        w = GlobalMaxPooling1D()(w)
        c = Embedding(len(char2idx)+1, 64)(in_c)
        c = Conv1D(64, 5, activation="relu")(c)
        c = GlobalMaxPooling1D()(c)
        x = Concatenate()([w,c])
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model([in_w, in_c], out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_fasttext_keras":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, FT_DIM, weights=[embedding_matrix], trainable=True, name="keras_ft_emb")(inp)
        x = Conv1D(128, 3, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_fasttext_gensim":
        inp = Input(shape=(MAX_WORD_LEN, FT_DIM), name="ft_seq_input")
        x = Conv1D(128, 3, activation="relu")(inp)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_word":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, 128)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_char":
        inp = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        x = Embedding(len(char2idx)+1, 64)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_combined":
        in_w = Input(shape=(MAX_WORD_LEN,), name="word_input")
        in_c = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        w = Embedding(VOCAB_SIZE, 128)(in_w)
        w = Bidirectional(LSTM(64))(w)
        c = Embedding(len(char2idx)+1, 64)(in_c)
        c = Bidirectional(LSTM(64))(c)
        x = Concatenate()([w,c])
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model([in_w, in_c], out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_fasttext_keras":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, FT_DIM, weights=[embedding_matrix], trainable=True)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_fasttext_gensim":
        inp = Input(shape=(MAX_WORD_LEN, FT_DIM), name="ft_seq_input")
        x = Bidirectional(LSTM(64))(inp)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    raise ValueError(f"Unsupported model: {model_name}")

# ------------------------- Train model (per-epoch) ---------------------------
# This function trains per-epoch so we can yield progress back to a caller.
def train_model_per_epoch(model_name, epochs=DEFAULT_EPOCHS, batch_size=16, lr=1e-3, verbose=False):
    model = build_model(model_name, lr=lr)
    full_history = {"loss":[], "val_loss":[], "accuracy":[], "val_accuracy":[]}

    # Select inputs
    if model_name in ["cnn_word","rnn_word","cnn_fasttext_keras","rnn_fasttext_keras"]:
        X_train = Xw_train
    elif model_name in ["cnn_char","rnn_char"]:
        X_train = Xc_train
    elif model_name in ["cnn_combined","rnn_combined"]:
        X_train = [Xw_train, Xc_train]
    elif model_name in ["cnn_fasttext_gensim","rnn_fasttext_gensim"]:
        X_train = Xft_train
    else:
        X_train = Xw_train

    for e in range(1, int(epochs)+1):
        # train for 1 epoch with validation_split=0.1
        hist = model.fit(X_train, y_train, validation_split=0.1, epochs=1, batch_size=int(batch_size), verbose=0)
        # aggregate
        for k,v in hist.history.items():
            if k not in full_history: full_history[k] = []
            full_history[k].append(v[0])
        # yield per-epoch metrics
        yield {
            "model": model_name,
            "epoch": e,
            "loss": float(full_history.get('loss')[-1]) if full_history.get('loss') else None,
            "val_loss": float(full_history.get('val_loss')[-1]) if full_history.get('val_loss') else None,
            "accuracy": float(full_history.get('accuracy')[-1]) if full_history.get('accuracy') else None,
            "val_accuracy": float(full_history.get('val_accuracy')[-1]) if full_history.get('val_accuracy') else None,
            "finished": False
        }

    # Save model at the end
    model_path = os.path.join(SAVED_MODELS_DIR, f"{model_name}.h5")
    model.save(model_path)

    # create final plot
    fig_path = os.path.join(PLOTS_DIR, f"{model_name}_train_plot.png")
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(full_history.get('loss',[]), label='train_loss')
    plt.plot(full_history.get('val_loss',[]), label='val_loss')
    plt.legend(); plt.title('Loss')
    plt.subplot(1,2,2)
    plt.plot(full_history.get('accuracy',[]), label='train_acc')
    plt.plot(full_history.get('val_accuracy',[]), label='val_acc')
    plt.legend(); plt.title('Accuracy')
    plt.tight_layout(); plt.savefig(fig_path); plt.close()

    # final yield marking finished
    yield {
        "model": model_name,
        "epoch": int(epochs),
        "loss": float(full_history.get('loss')[-1]) if full_history.get('loss') else None,
        "val_loss": float(full_history.get('val_loss')[-1]) if full_history.get('val_loss') else None,
        "accuracy": float(full_history.get('accuracy')[-1]) if full_history.get('accuracy') else None,
        "val_accuracy": float(full_history.get('val_accuracy')[-1]) if full_history.get('val_accuracy') else None,
        "finished": True,
        "model_path": model_path,
        "plot_path": fig_path,
        "history": full_history
    }

# ---------------------- Train ALL models generator --------------------------
# This generator trains each model sequentially and yields UI-updates for Gradio.
def train_all_models_generator(epochs=DEFAULT_EPOCHS, batch_size=16, lr=1e-3):
    # prepare summary list
    summary_rows = []
    # iterate
    for model_name in MODEL_NAMES:
        # Yield message that this model will start
        yield (f"Starting training: {model_name}", "", "", "", "", "")
        # loop per-epoch yields
        for epoch_info in train_model_per_epoch(model_name, epochs=epochs, batch_size=batch_size, lr=lr):
            # Compose train_info text and saved model listing (saved after finish)
            if not epoch_info.get('finished'):
                train_info = f"{epoch_info['model']} - epoch {epoch_info['epoch']}/{epochs} | acc={epoch_info['accuracy']:.4f} val_acc={epoch_info['val_accuracy']:.4f}"
                saved_list = "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')])
                plot_html = ""  # not yet
                decision = ""
                message = ""
                table_html = ""
                yield (train_info, saved_list, plot_html, decision, message, table_html)
            else:
                # finished - include plot and update metrics summary
                model_path = epoch_info.get('model_path')
                plot_path = epoch_info.get('plot_path')
                saved_list = "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')])
                train_info = f"Finished {model_name} (epochs={epochs}). Saved: {os.path.basename(model_path)}"
                plot_html = f"<img src='file://{plot_path}' width=800 />"
                # append summary
                summary_rows.append({
                    'model': model_name,
                    'final_accuracy': epoch_info.get('accuracy'),
                    'final_val_accuracy': epoch_info.get('val_accuracy'),
                    'loss': epoch_info.get('loss'),
                    'val_loss': epoch_info.get('val_loss'),
                    'model_file': os.path.basename(model_path),
                    'plot_file': os.path.basename(plot_path)
                })
                # save intermediate summary CSV
                pd.DataFrame(summary_rows).to_csv(METRICS_CSV, index=False)
                table_html = pd.DataFrame(summary_rows).to_html(index=False)
                decision = ""
                message = ""
                yield (train_info, saved_list, plot_html, decision, message, table_html)
    # All done - final message
    yield ("ALL MODELS TRAINED", "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')]), "", "", "", pd.read_csv(METRICS_CSV).to_html(index=False))

# ---------------- SBERT + phonetic + helpers (unchanged) -------------------
print("Loading SBERT (paraphrase-multilingual-MiniLM-L12-v2) \u2014 may take a moment ...")
try:
    sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
except Exception as e:
    print("Could not load SBERT model (offline?).", e)
    sbert = None

SBERT_DIM = sbert.get_sentence_embedding_dimension() if sbert is not None else 384

# phonetic and fuzzy

def phonetic_code(text):
    tr = unidecode(str(text))
    try:
        dm = jellyfish.double_metaphone(tr)[0]
    except:
        dm = jellyfish.metaphone(tr) if hasattr(jellyfish, "metaphone") else ""
    return dm or ""

def phonetic_score(a, b):
    ca = phonetic_code(a)
    cb = phonetic_code(b)
    if ca and cb and ca == cb:
        return 100.0
    return fuzz.token_set_ratio(unidecode(str(a)), unidecode(str(b)))

def cosine(a,b):
    a = np.array(a, dtype=float); b = np.array(b, dtype=float)
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na==0 or nb==0: return 0.0
    return float(np.dot(a,b)/(na*nb))

# embedding cache
emb_cache = {}

# helper to extract penultimate layer
def build_embedding_model_from_trained(model_obj, model_name):
    final_dense_idx = None
    for i, layer in enumerate(model_obj.layers[::-1]):
        if isinstance(layer, tf.keras.layers.Dense):
            # Check if output_shape attribute exists and is not None
            if hasattr(layer, 'output_shape') and layer.output_shape is not None:
                if layer.output_shape[-1] == 1:
                    final_dense_idx = len(model_obj.layers) - 1 - i
                    break
            # If output_shape is problematic, try to use layer.output.shape
            elif hasattr(layer, 'output') and tf.is_tensor(layer.output) and layer.output.shape is not None:
                 # Ensure the tensor shape is fully defined before accessing elements
                if len(layer.output.shape) > 0 and layer.output.shape[-1] == 1:
                    final_dense_idx = len(model_obj.layers) - 1 - i
                    break

    if final_dense_idx is None or final_dense_idx - 1 < 0:
        raise ValueError("Cannot find penultimate layer to extract embedding. Model structure might be unexpected.")
    embed_layer_output = model_obj.layers[final_dense_idx - 1].output
    return Model(inputs=model_obj.input, outputs=embed_layer_output)

# prepare embeddings for source
def prepare_embeddings_for_source(source_name):
    if source_name in emb_cache:
        return emb_cache[source_name]
    names = df['trade_name'].astype(str).tolist()
    embs = None
    if source_name == "bert_sentence":
        if sbert is None:
            raise ValueError("SBERT not loaded")
        embs = sbert.encode(names, convert_to_numpy=True, show_progress_bar=False)
    else:
        model_path = os.path.join(SAVED_MODELS_DIR, f"{source_name}.h5")
        if not os.path.exists(model_path):
            raise ValueError(f"Model '{source_name}' not found at {model_path}. Please train it first.")
        model_obj = load_model(model_path)
        emb_model = build_embedding_model_from_trained(model_obj, source_name)
        emb_list = []
        for name in names:
            if source_name in ["cnn_word", "rnn_word", "cnn_fasttext_keras", "rnn_fasttext_keras"]:
                x_input = np.array([encode_word(name)])
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_char", "rnn_char"]:
                x_input = np.array([encode_char(name)])
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_combined", "rnn_combined"]:
                x_input = [np.array([encode_word(name)]), np.array([encode_char(name)])]
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_fasttext_gensim", "rnn_fasttext_gensim"]:
                toks = str(name).split()[:MAX_WORD_LEN]
                seq = []
                for tkn in toks:
                    if tkn in fasttext_model.wv:
                        seq.append(fasttext_model.wv[tkn])
                    else:
                        seq.append(np.zeros(FT_DIM))
                while len(seq) < MAX_WORD_LEN:
                    seq.append(np.zeros(FT_DIM))
                x_input = np.array([seq])
                emb = emb_model.predict(x_input, verbose=0)[0]
            else:
                raise ValueError(f"Unknown source_name for embedding extraction: {source_name}")
            emb_list.append(emb)
        embs = np.vstack(emb_list)
    emb_cache[source_name] = (names, embs)
    return names, embs

# main lookup
def find_similar(query, source_name="bert_sentence", top_k=5, sim_threshold=85.0, require_registered_for_reject=True):
    query = str(query)
    q_emb = None
    if source_name == "bert_sentence":
        if sbert is None: raise ValueError("SBERT not loaded")
        q_emb = sbert.encode([query], convert_to_numpy=True)[0]
    else:
        model_path = os.path.join(SAVED_MODELS_DIR, f"{source_name}.h5")
        if not os.path.exists(model_path):
            raise ValueError(f"Model '{source_name}' not found at {model_path}. Please train it first.")
        model_obj = load_model(model_path)
        emb_model = build_embedding_model_from_trained(model_obj, source_name)
        if source_name in ["cnn_word", "rnn_word", "cnn_fasttext_keras", "rnn_fasttext_keras"]:
            x_input = np.array([encode_word(query)])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_char", "rnn_char"]:
            x_input = np.array([encode_char(query)])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_combined", "rnn_combined"]:
            x_input = [np.array([encode_word(query)]), np.array([encode_char(query)])]
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_fasttext_gensim", "rnn_fasttext_gensim"]:
            toks = str(query).split()[:MAX_WORD_LEN]
            seq = []
            for tkn in toks:
                if tkn in fasttext_model.wv:
                    seq.append(fasttext_model.wv[tkn])
                else:
                    seq.append(np.zeros(FT_DIM))
            while len(seq) < MAX_WORD_LEN:
                seq.append(np.zeros(FT_DIM))
            x_input = np.array([seq])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        else:
            raise ValueError(f"Unknown source_name for query embedding: {source_name}")

    names_list, embeddings_np = prepare_embeddings_for_source(source_name)
    results = []
    for idx, candidate in enumerate(names_list):
        emb = embeddings_np[idx]
        sim = cosine(q_emb, emb) * 100.0
        fuzzy_ratio = fuzz.token_set_ratio(query, candidate)
        phon_score = phonetic_score(query, candidate)
        reg_status = df.loc[df["trade_name"] == candidate, "registration_status"]
        reg_status = reg_status.values[0] if len(reg_status) > 0 else ""
        results.append({
            "trade_name": candidate,
            "registration_status": reg_status,
            "embedding_similarity_pct": round(sim, 2),
            "fuzzy_pct": round(fuzzy_ratio, 2),
            "phonetic_pct": round(phon_score, 2)
        })
    results = sorted(results, key=lambda x: x["embedding_similarity_pct"], reverse=True)
    top_k_results = results[:top_k]
    reject_reasons = []
    for r in top_k_results:
        if r["embedding_similarity_pct"] >= sim_threshold:
            if require_registered_for_reject:
                if str(r["registration_status"]).strip() == "የተመዘገበ":
                    reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
            else:
                reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
    decision = "ACCEPTED"
    message = f"No registered names \u2265 {sim_threshold:.1f}% similarity found."
    if len(reject_reasons) > 0:
        decision = "REJECTED"
        message = f"Found {len(reject_reasons)} registered name(s) \u2265 {sim_threshold:.1f}% similarity."
    out_df = pd.DataFrame(top_k_results)
    return decision, message, out_df

# -------------------- Utility: list saved models -----------------------------
def list_saved_models():
    files = [f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')]
    return files

# ------------------------- Gradio wiring ------------------------------------
# Single-model training wrapper (non-streaming) kept for compatibility

def ui_train_single_and_check(
    chosen_model, epochs, batch_size, lr, train_flag,
    check_query, source_for_check, top_k, sim_threshold, require_registered
):
    train_info = ""
    plot_html = ""
    saved_list = "<br>".join(list_saved_models())
    if train_flag:
        try:
            # run full training (this will block until finish)
            final = None
            for epoch_info in train_model_per_epoch(chosen_model, epochs=epochs, batch_size=batch_size, lr=lr):
                final = epoch_info
            train_info = f"Trained {chosen_model}: epochs={epochs}, batch={batch_size}, lr={lr}"
            if final and final.get('plot_path'):
                plot_html = f"<img src='file://{final.get('plot_path')}' width=800 />"
            saved_list = "<br>".join(list_saved_models())
            if chosen_model in emb_cache: del emb_cache[chosen_model]
        except Exception as e:
            train_info = "TRAIN ERROR: " + str(e)

    decision = ""
    message = ""
    table_html = ""
    if str(check_query).strip():
        try:
            decision, message, out_df = find_similar(check_query, source_name=source_for_check, top_k=int(top_k), sim_threshold=float(sim_threshold), require_registered_for_reject=bool(require_registered))
            table_html = out_df.to_html(index=False)
        except Exception as e:
            decision = "ERROR"
            message = str(e)
            table_html = ""
    return train_info, saved_list, plot_html, decision, message, table_html

# Train-all generator wrapper for Gradio streaming
def ui_train_all(epochs, batch_size, lr):
    # This is a generator that yields tuples matching our UI outputs
    for out in train_all_models_generator(epochs=int(epochs), batch_size=int(batch_size), lr=float(lr)):
        # out is tuple (train_info, saved_list, plot_html, decision, message, table_html)
        yield out

# Build Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## Trade Name Similarity System \u2014 Multi-Model Trainer\n\nUse 'Train All Models' to train every model sequentially and stream per-epoch progress.\nUse 'Train Single Model' to train only the selected model.")

    # Define a hidden checkbox component to control training behavior for check_btn
    # This replaces the incorrect direct passing of `False`.
    no_train_checkbox_input = gr.Checkbox(value=False, visible=False, label="Do not train (hidden input)")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Training Controls")
            chosen_model = gr.Dropdown(MODEL_NAMES, value="cnn_word", label="Choose model to train")
            epochs = gr.Slider(minimum=1, maximum=50, step=1, value=5, label="Epochs")
            batch_size = gr.Slider(minimum=4, maximum=64, step=1, value=16, label="Batch size")
            lr = gr.Number(value=1e-3, label="Learning rate")
            train_flag = gr.Checkbox(label="Train model now?", value=False)
            train_btn = gr.Button("Run Training")
            train_all_btn = gr.Button("Train ALL Models (stream)")
            train_info_out = gr.Textbox(label="Train Info", interactive=False)
            saved_models_out = gr.HTML(label="Saved Models")
            train_plot_out = gr.HTML(label="Training Plot")

        with gr.Column(scale=1):
            gr.Markdown("### Similarity Check")
            check_query = gr.Textbox(lines=2, label="Input trade name to check")
            source_for_check_options = MODEL_NAMES + ["bert_sentence"]
            source_for_check = gr.Dropdown(source_for_check_options, value="bert_sentence", label="Source for Similarity Check")
            top_k = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="Top K candidates")
            sim_threshold = gr.Slider(minimum=50, maximum=100, step=1, value=85, label="Reject threshold (embedding %)")
            require_registered = gr.Checkbox(label="Only reject if matching name is registered (የተመዘገበ)", value=True)
            check_btn = gr.Button("Check Similarity")
            decision_out = gr.Textbox(label="Decision", interactive=False)
            message_out = gr.Textbox(label="Message", interactive=False)
            table_out = gr.HTML(label="Similarity Results")

    # Single model training (blocking)
    train_btn.click(
        fn=ui_train_single_and_check,
        inputs=[
            chosen_model, epochs, batch_size, lr, train_flag,
            check_query, source_for_check, top_k, sim_threshold, require_registered
        ],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

    # Train all models (streaming generator) -> will update the same outputs incrementally
    train_all_btn.click(
        fn=ui_train_all,
        inputs=[epochs, batch_size, lr],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

    # Similarity check (non-training)
    check_btn.click(
        fn=ui_train_single_and_check,
        inputs=[
            chosen_model, epochs, batch_size, lr,
            no_train_checkbox_input, # Pass the hidden checkbox component directly
            check_query, source_for_check, top_k, sim_threshold, require_registered
        ],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

# Launch (you can set share=True in Colab to obtain public link)
demo.launch(share=True)

Loading SBERT (paraphrase-multilingual-MiniLM-L12-v2) — may take a moment ...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90b4e565b85861fd4a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Multi-Model Trainer + Per-epoch Progress Streaming + SBERT + Phonetic + Gradio UI
# Single-file ready for Colab or local notebook. This enhances your original script to:
# - Train *all* models sequentially
# - Stream per-epoch progress back to Gradio via a generator (so the UI updates live)
# - Save per-model training plots and a summary CSV containing final metrics for comparison
# - Provide two Gradio actions: Train single model (as before) and Train ALL models (streaming)

# NOTE: run this entire file in one Colab cell (or a Python environment with display support).
# Make sure you mount Google Drive if you want saved models under /content/drive.

# --------- Install required packages (uncomment if running first time) ---------
# !pip install -q sentence-transformers gradio gensim annoy jellyfish unidecode rapidfuzz matplotlib

# ------------------------------ Imports -------------------------------------
import os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, LSTM, Bidirectional, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sentence_transformers import SentenceTransformer
from gensim.models import FastText
from annoy import AnnoyIndex
from rapidfuzz import fuzz
import jellyfish
from unidecode import unidecode
import gradio as gr
import pickle

# ------------------------------ Config --------------------------------------
MODEL_NAMES = [
    "cnn_word","cnn_char","cnn_combined",
    "cnn_fasttext_keras","cnn_fasttext_gensim",
    "rnn_word","rnn_char","rnn_combined",
    "rnn_fasttext_keras","rnn_fasttext_gensim"
]
DEFAULT_EPOCHS = 5
FT_DIM = 50
KERAS_EMB_DIM = 64
SAVED_MODELS_DIR = "/content/drive/MyDrive/TradeNameSimilarity/Models"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
PLOTS_DIR = "/content/drive/MyDrive/TradeNameSimilarity/Models/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)
METRICS_CSV = os.path.join(PLOTS_DIR, "all_models_summary.csv")

# ----------------------------- Sample data ----------------------------------
sample_data = {
    "trade_name": [
        "ማኑፋክቸሪንግ ካምፓኒ",
        "አፍሪእሸቱ አ.ማ",
        "አብሮአዲስ የህብረት ስራ ማህበር",
        "ቴክሱራፌል ሶልዩሽን",
        "አፍሪእናተ ሶልዩሽን",
        "አፍሪያህንፃ አክሲዮን ማህበር",
        "ኢትዮተቋራጭ አ.ማ",
        "ሰነእዚህ ቢዝነስ ሴንተር",
        "በዚህ ሰርቪስ",
        "አዲስ ንግድ ስም"
    ],
    "registration_status": [
        "የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ",
        "የተመዘገበ","የተመዘገበ","የተመዘገበ","የተመዘገበ","አልተመዘገበ"
    ]
}

df = pd.DataFrame(sample_data)
df['label'] = df['registration_status'].apply(lambda x: 1 if str(x).strip()=="የተመዘገበ" else 0)

# --------------------------- Preprocessing ----------------------------------
texts = df['trade_name'].astype(str).tolist()

# WORD tokenizer
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(texts)
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1
MAX_WORD_LEN = 20

def encode_word(s):
    seq = tokenizer.texts_to_sequences([str(s)])
    return pad_sequences(seq, maxlen=MAX_WORD_LEN)[0]

# CHAR mapping
all_chars = sorted(list(set(" ".join(texts))))
char2idx = {c:i+1 for i,c in enumerate(all_chars)}
MAX_CHAR_LEN = 40

def encode_char(s):
    s = str(s)[:MAX_CHAR_LEN]
    arr = [char2idx.get(c,0) for c in s]
    if len(arr) < MAX_CHAR_LEN:
        arr += [0]*(MAX_CHAR_LEN - len(arr))
    return np.array(arr)

# FastText (gensim) training
ft_sentences = [t.split() for t in texts]
fasttext_model = FastText(sentences=ft_sentences, vector_size=FT_DIM, window=3, min_count=1, epochs=20)

# Pre-init embedding matrix for Keras-fasttext option
embedding_matrix = np.zeros((VOCAB_SIZE, FT_DIM))
for word, i in word_index.items():
    if i < VOCAB_SIZE:
        if word in fasttext_model.wv:
            embedding_matrix[i] = fasttext_model.wv[word]
        else:
            embedding_matrix[i] = np.random.uniform(-0.25, 0.25, FT_DIM)

# Gensim FT seq builder
def texts_to_ft_seq_array(texts_list):
    seqs = []
    for t in texts_list:
        toks = str(t).split()[:MAX_WORD_LEN]
        vecs = []
        for token in toks:
            if token in fasttext_model.wv:
                vecs.append(fasttext_model.wv[token])
            else:
                vecs.append(np.zeros(FT_DIM))
        while len(vecs) < MAX_WORD_LEN:
            vecs.append(np.zeros(FT_DIM))
        seqs.append(np.array(vecs))
    return np.array(seqs)

# Precompute inputs
X_word = np.array([encode_word(t) for t in texts])
X_char = np.array([encode_char(t) for t in texts])
X_ft_seqs = texts_to_ft_seq_array(texts)
y = df['label'].values

Xw_train, Xw_test, Xc_train, Xc_test, Xft_train, Xft_test, y_train, y_test = train_test_split(
    X_word, X_char, X_ft_seqs, y, test_size=0.2, random_state=42
)

# ---------------------------- Model factory ---------------------------------
def build_model(model_name, lr=1e-3):
    if model_name == "cnn_word":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, 128)(inp)
        x = Conv1D(128, 3, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_char":
        inp = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        x = Embedding(len(char2idx)+1, 64)(inp)
        x = Conv1D(128, 5, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_combined":
        in_w = Input(shape=(MAX_WORD_LEN,), name="word_input")
        in_c = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        w = Embedding(VOCAB_SIZE, 128)(in_w)
        w = Conv1D(64, 3, activation="relu")(w)
        w = GlobalMaxPooling1D()(w)
        c = Embedding(len(char2idx)+1, 64)(in_c)
        c = Conv1D(64, 5, activation="relu")(c)
        c = GlobalMaxPooling1D()(c)
        x = Concatenate()([w,c])
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model([in_w, in_c], out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_fasttext_keras":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, FT_DIM, weights=[embedding_matrix], trainable=True, name="keras_ft_emb")(inp)
        x = Conv1D(128, 3, activation="relu")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "cnn_fasttext_gensim":
        inp = Input(shape=(MAX_WORD_LEN, FT_DIM), name="ft_seq_input")
        x = Conv1D(128, 3, activation="relu")(inp)
        x = GlobalMaxPooling1D()(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_word":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, 128)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_char":
        inp = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        x = Embedding(len(char2idx)+1, 64)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_combined":
        in_w = Input(shape=(MAX_WORD_LEN,), name="word_input")
        in_c = Input(shape=(MAX_CHAR_LEN,), name="char_input")
        w = Embedding(VOCAB_SIZE, 128)(in_w)
        w = Bidirectional(LSTM(64))(w)
        c = Embedding(len(char2idx)+1, 64)(in_c)
        c = Bidirectional(LSTM(64))(c)
        x = Concatenate()([w,c])
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model([in_w, in_c], out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_fasttext_keras":
        inp = Input(shape=(MAX_WORD_LEN,), name="word_input")
        x = Embedding(VOCAB_SIZE, FT_DIM, weights=[embedding_matrix], trainable=True)(inp)
        x = Bidirectional(LSTM(64))(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    if model_name == "rnn_fasttext_gensim":
        inp = Input(shape=(MAX_WORD_LEN, FT_DIM), name="ft_seq_input")
        x = Bidirectional(LSTM(64))(inp)
        x = Dense(64, activation="relu")(x)
        out = Dense(1, activation="sigmoid")(x)
        m = Model(inp, out, name=model_name)
        m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
        return m

    raise ValueError(f"Unsupported model: {model_name}")

# ------------------------- Train model (per-epoch) ---------------------------
# This function trains per-epoch so we can yield progress back to a caller.
def evaluate_on_test(model_obj, model_name):
    """Evaluate trained model on the holdout test set and return metrics dict."""
    # select test inputs depending on model type
    if model_name in ["cnn_word","rnn_word","cnn_fasttext_keras","rnn_fasttext_keras"]:
        X_test_local = Xw_test
    elif model_name in ["cnn_char","rnn_char"]:
        X_test_local = Xc_test
    elif model_name in ["cnn_combined","rnn_combined"]:
        X_test_local = [Xw_test, Xc_test]
    elif model_name in ["cnn_fasttext_gensim","rnn_fasttext_gensim"]:
        X_test_local = Xft_test
    else:
        X_test_local = Xw_test

    # predict probabilities
    preds_proba = model_obj.predict(X_test_local, verbose=0)
    # ensure shape
    preds_proba = np.array(preds_proba).reshape(-1)
    preds = (preds_proba >= 0.5).astype(int)

    acc = float(accuracy_score(y_test, preds)) if len(y_test) > 0 else 0.0
    prec = float(precision_score(y_test, preds, zero_division=0))
    rec = float(recall_score(y_test, preds, zero_division=0))
    f1 = float(f1_score(y_test, preds, zero_division=0))
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


def train_model_per_epoch(model_name, epochs=DEFAULT_EPOCHS, batch_size=16, lr=1e-3, verbose=False):
    model = build_model(model_name, lr=lr)
    full_history = {"loss":[], "val_loss":[], "accuracy":[], "val_accuracy":[]}

    # Select inputs
    if model_name in ["cnn_word","rnn_word","cnn_fasttext_keras","rnn_fasttext_keras"]:
        X_train = Xw_train
    elif model_name in ["cnn_char","rnn_char"]:
        X_train = Xc_train
    elif model_name in ["cnn_combined","rnn_combined"]:
        X_train = [Xw_train, Xc_train]
    elif model_name in ["cnn_fasttext_gensim","rnn_fasttext_gensim"]:
        X_train = Xft_train
    else:
        X_train = Xw_train

    for e in range(1, int(epochs)+1):
        # train for 1 epoch with validation_split=0.1
        hist = model.fit(X_train, y_train, validation_split=0.1, epochs=1, batch_size=int(batch_size), verbose=0)
        # aggregate
        for k,v in hist.history.items():
            if k not in full_history: full_history[k] = []
            full_history[k].append(v[0])
        # yield per-epoch metrics
        yield {
            "model": model_name,
            "epoch": e,
            "loss": float(full_history.get('loss')[-1]) if full_history.get('loss') else None,
            "val_loss": float(full_history.get('val_loss')[-1]) if full_history.get('val_loss') else None,
            "accuracy": float(full_history.get('accuracy')[-1]) if full_history.get('accuracy') else None,
            "val_accuracy": float(full_history.get('val_accuracy')[-1]) if full_history.get('val_accuracy') else None,
            "finished": False
        }

    # Save model at the end
    model_path = os.path.join(SAVED_MODELS_DIR, f"{model_name}.h5")
    model.save(model_path)

    # Evaluate on holdout test set
    metrics = evaluate_on_test(model, model_name)

    # final yield marking finished (no plot generation)
    yield {
        "model": model_name,
        "epoch": int(epochs),
        "loss": float(full_history.get('loss')[-1]) if full_history.get('loss') else None,
        "val_loss": float(full_history.get('val_loss')[-1]) if full_history.get('val_loss') else None,
        "accuracy": float(full_history.get('accuracy')[-1]) if full_history.get('accuracy') else None,
        "val_accuracy": float(full_history.get('val_accuracy')[-1]) if full_history.get('val_accuracy') else None,
        "finished": True,
        "model_path": model_path,
        "metrics": metrics,
        "history": full_history
    }

# ---------------------- Train ALL models generator --------------------------
# This generator trains each model sequentially and yields UI-updates for Gradio.
def train_all_models_generator(epochs=DEFAULT_EPOCHS, batch_size=16, lr=1e-3):
    # prepare summary list
    summary_rows = []
    # iterate
    for model_name in MODEL_NAMES:
        # Yield message that this model will start
        yield (f"Starting training: {model_name}", "", "", "", "", "")
        # loop per-epoch yields
        for epoch_info in train_model_per_epoch(model_name, epochs=epochs, batch_size=batch_size, lr=lr):
            # Compose train_info text and saved model listing (saved after finish)
            if not epoch_info.get('finished'):
                train_info = f"{epoch_info['model']} - epoch {epoch_info['epoch']}/{epochs} | acc={epoch_info.get('accuracy'):.4f} val_acc={epoch_info.get('val_accuracy'):.4f}"
                saved_list = "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')])
                metrics_html = ""
                decision = ""
                message = ""
                table_html = ""
                yield (train_info, saved_list, metrics_html, decision, message, table_html)
            else:
                # finished - include metrics and update metrics summary
                model_path = epoch_info.get('model_path')
                saved_list = "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')])
                train_info = f"Finished {model_name} (epochs={epochs}). Saved: {os.path.basename(model_path)}"
                metrics = epoch_info.get('metrics', {})
                # append summary
                summary_rows.append({
                    'model': model_name,
                    'final_accuracy': metrics.get('accuracy'),
                    'final_precision': metrics.get('precision'),
                    'final_recall': metrics.get('recall'),
                    'final_f1': metrics.get('f1'),
                    'loss': epoch_info.get('loss'),
                    'val_loss': epoch_info.get('val_loss'),
                    'model_file': os.path.basename(model_path)
                })
                # save intermediate summary CSV
                pd.DataFrame(summary_rows).to_csv(METRICS_CSV, index=False)
                table_html = pd.DataFrame(summary_rows).to_html(index=False)
                metrics_html = table_html
                decision = ""
                message = ""
                yield (train_info, saved_list, metrics_html, decision, message, table_html)
    # All done - final message
    final_html = pd.read_csv(METRICS_CSV).to_html(index=False) if os.path.exists(METRICS_CSV) else ""
    yield ("ALL MODELS TRAINED", "<br>".join([f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')]), "", "", "", final_html)

# ---------------- SBERT + phonetic + helpers (unchanged) -------------------
print("Loading SBERT (paraphrase-multilingual-MiniLM-L12-v2) — may take a moment ...")
try:
    sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
except Exception as e:
    print("Could not load SBERT model (offline?).", e)
    sbert = None

SBERT_DIM = sbert.get_sentence_embedding_dimension() if sbert is not None else 384

# phonetic and fuzzy

def phonetic_code(text):
    tr = unidecode(str(text))
    try:
        dm = jellyfish.double_metaphone(tr)[0]
    except:
        dm = jellyfish.metaphone(tr) if hasattr(jellyfish, "metaphone") else ""
    return dm or ""

def phonetic_score(a, b):
    ca = phonetic_code(a)
    cb = phonetic_code(b)
    if ca and cb and ca == cb:
        return 100.0
    return fuzz.token_set_ratio(unidecode(str(a)), unidecode(str(b)))

def cosine(a,b):
    a = np.array(a, dtype=float); b = np.array(b, dtype=float)
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na==0 or nb==0: return 0.0
    return float(np.dot(a,b)/(na*nb))

# embedding cache
emb_cache = {}

# helper to extract penultimate layer
def build_embedding_model_from_trained(model_obj, model_name):
    final_dense_idx = None
    for i, layer in enumerate(model_obj.layers[::-1]):
        if isinstance(layer, tf.keras.layers.Dense) and layer.output_shape[-1] == 1:
            final_dense_idx = len(model_obj.layers) - 1 - i
            break
    if final_dense_idx is None or final_dense_idx - 1 < 0:
        raise ValueError("Cannot find penultimate layer to extract embedding. Model structure might be unexpected.")
    embed_layer_output = model_obj.layers[final_dense_idx - 1].output
    return Model(inputs=model_obj.input, outputs=embed_layer_output)

# prepare embeddings for source
def prepare_embeddings_for_source(source_name):
    if source_name in emb_cache:
        return emb_cache[source_name]
    names = df['trade_name'].astype(str).tolist()
    embs = None
    if source_name == "bert_sentence":
        if sbert is None:
            raise ValueError("SBERT not loaded")
        embs = sbert.encode(names, convert_to_numpy=True, show_progress_bar=False)
    else:
        model_path = os.path.join(SAVED_MODELS_DIR, f"{source_name}.h5")
        if not os.path.exists(model_path):
            raise ValueError(f"Model '{source_name}' not found at {model_path}. Please train it first.")
        model_obj = load_model(model_path)
        emb_model = build_embedding_model_from_trained(model_obj, source_name)
        emb_list = []
        for name in names:
            if source_name in ["cnn_word", "rnn_word", "cnn_fasttext_keras", "rnn_fasttext_keras"]:
                x_input = np.array([encode_word(name)])
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_char", "rnn_char"]:
                x_input = np.array([encode_char(name)])
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_combined", "rnn_combined"]:
                x_input = [np.array([encode_word(name)]), np.array([encode_char(name)])]
                emb = emb_model.predict(x_input, verbose=0)[0]
            elif source_name in ["cnn_fasttext_gensim", "rnn_fasttext_gensim"]:
                toks = str(name).split()[:MAX_WORD_LEN]
                seq = []
                for tkn in toks:
                    if tkn in fasttext_model.wv:
                        seq.append(fasttext_model.wv[tkn])
                    else:
                        seq.append(np.zeros(FT_DIM))
                while len(seq) < MAX_WORD_LEN:
                    seq.append(np.zeros(FT_DIM))
                x_input = np.array([seq])
                emb = emb_model.predict(x_input, verbose=0)[0]
            else:
                raise ValueError(f"Unknown source_name for embedding extraction: {source_name}")
            emb_list.append(emb)
        embs = np.vstack(emb_list)
    emb_cache[source_name] = (names, embs)
    return names, embs

# main lookup
def find_similar(query, source_name="bert_sentence", top_k=5, sim_threshold=85.0, require_registered_for_reject=True):
    query = str(query)
    q_emb = None
    if source_name == "bert_sentence":
        if sbert is None: raise ValueError("SBERT not loaded")
        q_emb = sbert.encode([query], convert_to_numpy=True)[0]
    else:
        model_path = os.path.join(SAVED_MODELS_DIR, f"{source_name}.h5")
        if not os.path.exists(model_path):
            raise ValueError(f"Model '{source_name}' not found at {model_path}. Please train it first.")
        model_obj = load_model(model_path)
        emb_model = build_embedding_model_from_trained(model_obj, source_name)
        if source_name in ["cnn_word", "rnn_word", "cnn_fasttext_keras", "rnn_fasttext_keras"]:
            x_input = np.array([encode_word(query)])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_char", "rnn_char"]:
            x_input = np.array([encode_char(query)])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_combined", "rnn_combined"]:
            x_input = [np.array([encode_word(query)]), np.array([encode_char(query)])]
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        elif source_name in ["cnn_fasttext_gensim", "rnn_fasttext_gensim"]:
            toks = str(query).split()[:MAX_WORD_LEN]
            seq = []
            for tkn in toks:
                if tkn in fasttext_model.wv:
                    seq.append(fasttext_model.wv[tkn])
                else:
                    seq.append(np.zeros(FT_DIM))
            while len(seq) < MAX_WORD_LEN:
                seq.append(np.zeros(FT_DIM))
            x_input = np.array([seq])
            q_emb = emb_model.predict(x_input, verbose=0)[0]
        else:
            raise ValueError(f"Unknown source_name for query embedding: {source_name}")

    names_list, embeddings_np = prepare_embeddings_for_source(source_name)
    results = []
    for idx, candidate in enumerate(names_list):
        emb = embeddings_np[idx]
        sim = cosine(q_emb, emb) * 100.0
        fuzzy_ratio = fuzz.token_set_ratio(query, candidate)
        phon_score = phonetic_score(query, candidate)
        reg_status = df.loc[df["trade_name"] == candidate, "registration_status"]
        reg_status = reg_status.values[0] if len(reg_status) > 0 else ""
        results.append({
            "trade_name": candidate,
            "registration_status": reg_status,
            "embedding_similarity_pct": round(sim, 2),
            "fuzzy_pct": round(fuzzy_ratio, 2),
            "phonetic_pct": round(phon_score, 2)
        })
    results = sorted(results, key=lambda x: x["embedding_similarity_pct"], reverse=True)
    top_k_results = results[:top_k]
    reject_reasons = []
    for r in top_k_results:
        if r["embedding_similarity_pct"] >= sim_threshold:
            if require_registered_for_reject:
                if str(r["registration_status"]).strip() == "የተመዘገበ":
                    reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
            else:
                reject_reasons.append((r["trade_name"], r["embedding_similarity_pct"]))
    decision = "ACCEPTED"
    message = f"No registered names ≥ {sim_threshold:.1f}% similarity found."
    if len(reject_reasons) > 0:
        decision = "REJECTED"
        message = f"Found {len(reject_reasons)} registered name(s) ≥ {sim_threshold:.1f}% similarity."
    out_df = pd.DataFrame(top_k_results)
    return decision, message, out_df

# -------------------- Utility: list saved models -----------------------------
def list_saved_models():
    files = [f for f in os.listdir(SAVED_MODELS_DIR) if f.endswith('.h5')]
    return files

# ------------------------- Gradio wiring ------------------------------------
# Single-model training wrapper (non-streaming) kept for compatibility

def ui_train_single_and_check(
    chosen_model, epochs, batch_size, lr, train_flag,
    check_query, source_for_check, top_k, sim_threshold, require_registered
):
    train_info = ""
    metrics_html = ""
    saved_list = "<br>".join(list_saved_models())
    if train_flag:
        try:
            # run full training (this will block until finish)
            final = None
            for epoch_info in train_model_per_epoch(chosen_model, epochs=epochs, batch_size=batch_size, lr=lr):
                final = epoch_info
            train_info = f"Trained {chosen_model}: epochs={epochs}, batch={batch_size}, lr={lr}"
            if final and final.get('metrics'):
                metrics = final.get('metrics')
                metrics_html = pd.DataFrame([{"metric":k, "value":v} for k,v in metrics.items()]).to_html(index=False)
            saved_list = "<br>".join(list_saved_models())
            if chosen_model in emb_cache: del emb_cache[chosen_model]
        except Exception as e:
            train_info = "TRAIN ERROR: " + str(e)

    decision = ""
    message = ""
    table_html = ""
    if str(check_query).strip():
        try:
            decision, message, out_df = find_similar(check_query, source_name=source_for_check, top_k=int(top_k), sim_threshold=float(sim_threshold), require_registered_for_reject=bool(require_registered))
            table_html = out_df.to_html(index=False)
        except Exception as e:
            decision = "ERROR"
            message = str(e)
            table_html = ""
    return train_info, saved_list, metrics_html, decision, message, table_html

# Train-all generator wrapper for Gradio streaming
def ui_train_all(epochs, batch_size, lr):
    # This is a generator that yields tuples matching our UI outputs
    for out in train_all_models_generator(epochs=int(epochs), batch_size=int(batch_size), lr=float(lr)):
        # out is tuple (train_info, saved_list, plot_html, decision, message, table_html)
        yield out

# Build Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## Trade Name Similarity System — Multi-Model Trainer\n\nUse 'Train All Models' to train every model sequentially and stream per-epoch progress.\nUse 'Train Single Model' to train only the selected model.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Training Controls")
            chosen_model = gr.Dropdown(MODEL_NAMES, value="cnn_word", label="Choose model to train")
            epochs = gr.Slider(minimum=1, maximum=50, step=1, value=5, label="Epochs")
            batch_size = gr.Slider(minimum=4, maximum=64, step=1, value=16, label="Batch size")
            lr = gr.Number(value=1e-3, label="Learning rate")
            train_flag = gr.Checkbox(label="Train model now?", value=False)
            train_btn = gr.Button("Run Training")
            train_all_btn = gr.Button("Train ALL Models (stream)")
            train_info_out = gr.Textbox(label="Train Info", interactive=False)
            saved_models_out = gr.HTML(label="Saved Models")
            train_plot_out = gr.HTML(label="Training Plot")

        with gr.Column(scale=1):
            gr.Markdown("### Similarity Check")
            check_query = gr.Textbox(lines=2, label="Input trade name to check")
            source_for_check_options = MODEL_NAMES + ["bert_sentence"]
            source_for_check = gr.Dropdown(source_for_check_options, value="bert_sentence", label="Source for Similarity Check")
            top_k = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="Top K candidates")
            sim_threshold = gr.Slider(minimum=50, maximum=100, step=1, value=85, label="Reject threshold (embedding %)")
            require_registered = gr.Checkbox(label="Only reject if candidate is registered (የተመዘገበ)", value=True)
            check_btn = gr.Button("Check Similarity")
            decision_out = gr.Textbox(label="Decision", interactive=False)
            message_out = gr.Textbox(label="Message", interactive=False)
            table_out = gr.HTML(label="Similarity Results")

    # Single model training (blocking)
    train_btn.click(
        fn=ui_train_single_and_check,
        inputs=[
            chosen_model, epochs, batch_size, lr,
            check_query, source_for_check, top_k, sim_threshold, require_registered
        ],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

    # Train all models (streaming generator) -> will update the same outputs incrementally
    train_all_btn.click(
        fn=ui_train_all,
        inputs=[epochs, batch_size, lr],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

    # Similarity check (non-training)
    check_btn.click(
        fn=ui_train_single_and_check,
        inputs=[
            chosen_model, epochs, batch_size, lr,
             # ensure train_flag False
            check_query, source_for_check, top_k, sim_threshold, require_registered
        ],
        outputs=[train_info_out, saved_models_out, train_plot_out, decision_out, message_out, table_out]
    )

# Launch (you can set share=True in Colab to obtain public link)
demo.launch(share=True)

# --- Model Loading Fix Added Below ---
model_name_map = {
    "CNN_Char": "cnn_char_emb.h5",
    "SBERT": "sbert_model.h5",
    "TFIDF": "tfidf_model.pkl"
}

model_filename = model_name_map.get(chosen_model)
if not model_filename:
    return f"Unknown model '{chosen_model}'. Please train it first.", None

model_path = f"/content/drive/MyDrive/TradeNameSimilarity/Models/{model_filename}"
if not os.path.exists(model_path):
    return f"Model '{chosen_model}' not found at {model_path}. Please train it first.", None

